# Task 3: queue worker on Colab

Runs top to bottom on a CPU runtime and ends in a cell that **blocks while the worker serves the
Drive queue** (`su2queue/jobs/*.json` in, `su2queue/runs/<id>/` out). Every setup cell skips work
that is already done; set `FORCE = True` in the first code cell to redo downloads.
Layout and flags: `docs/specs/01_interfaces.md`, "Queue transport" and "Job directory".

## 0. Environment
Prints the core count and RAM. Look at: `nproc` (Colab gives 2 or 8 vCPUs) and the total memory.

In [ ]:
FORCE = False   # True: redo downloads even if their outputs exist
!nproc
!free -h

## 1. Repository
Clones the repo, checks out `COMMIT` (a branch, tag or sha), installs it, prints the exact commit.
Look at: the printed sha. The worker compares it with the `code_commit` in each job file, and a
different commit can produce a different mesh or config (the job then fails as `mesh_mismatch`).
After pulling a new commit, restart the runtime (Runtime -> Restart runtime) before running: Python
keeps the old modules in memory otherwise. The cell refuses to run if repo modules are already loaded.

In [ ]:
COMMIT = "main"   # branch, tag or commit sha
import os, sys, subprocess
if not os.path.isdir("/content/ai-engineering"):
    !git clone -q https://github.com/emarinhoss/ai-engineering.git /content/ai-engineering
%cd /content/ai-engineering
# Modules already imported by this kernel would not see a new commit: refuse to continue.
stale = [m for m in sys.modules if m.split(".")[0] in ("physics", "geometry", "cases", "postprocess", "worker", "mcp_server")]
assert not stale, f"repo modules already imported in this kernel ({stale[:3]}...): Runtime -> Restart runtime, then run again"
!git fetch -q --all
!git checkout -q {COMMIT}
!git pull -q --ff-only
!pip install -q -e .
sys.path.insert(0, "/content/ai-engineering")
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("commit", HEAD)
assert HEAD == subprocess.check_output(["git", "rev-parse", f"origin/{COMMIT}"], text=True).strip(), "local checkout is not at the remote commit"

## 2. SU2 v8.2.0 binaries (serial and MPI) and the solver command
Downloads both release variants and unzips them (each zip contains a second zip with `bin/`), installs
OpenMPI only if `mpirun` is missing, records the sha256 of each zip and the first line of
`SU2_CFD --help`. Look at: the two version lines in the printed JSON (they must say 8.2.0).
Colab runs as root, hence `--allow-run-as-root`.

**Number of MPI ranks.** `N_RANKS = max(1, os.cpu_count() // 2)`: on Colab the vCPU count can be 2 or 8
and hyperthreads give little. The Task 1 timing gave a 1.12x speed-up with 2 ranks on 8 vCPUs, so more
ranks are not expected to help much; the optional scaling test (section 4) measures it for this
runtime. `SU2_CMD` is the MPI command; if a 5-second `mpirun -n 2 ... SU2_CFD --help` test fails,
or `N_RANKS` is 1, the serial binary is used instead. Look at: the printed `SU2_CMD`.

In [ ]:
import hashlib, json, shutil
os.environ.update(OMPI_ALLOW_RUN_AS_ROOT="1", OMPI_ALLOW_RUN_AS_ROOT_CONFIRM="1")
if not shutil.which("mpirun"):
    !apt-get -qq install -y openmpi-bin
REL = "https://github.com/su2code/SU2/releases/download/v8.2.0"
SU2 = {"serial": ("SU2-v8.2.0-linux64.zip", "/content/su2"),
       "mpi": ("SU2-v8.2.0-linux64-mpi.zip", "/content/su2mpi")}
versions = {}
for kind, (zipname, root) in SU2.items():
    zp = f"/content/{zipname}"
    if FORCE or not os.path.exists(zp):
        !wget -q -O {zp} {REL}/{zipname}
    if FORCE or not os.path.exists(f"{root}/bin/SU2_CFD"):
        # The release zip holds a second zip (linux64.zip) that holds bin/; unzip both.
        !mkdir -p {root} && unzip -q -o {zp} -d {root}
        for inner in [f for f in os.listdir(root) if f.endswith(".zip")]:
            !unzip -q -o {root}/{inner} -d {root}
    assert os.path.exists(f"{root}/bin/SU2_CFD"), f"no bin/SU2_CFD in {root}; run !find {root} -name SU2_CFD"
    !chmod +x {root}/bin/*
    out = subprocess.run([f"{root}/bin/SU2_CFD", "--help"], capture_output=True, text=True, timeout=60, check=False)
    first = next((l.strip() for l in (out.stdout + out.stderr).splitlines() if l.strip()), "")
    versions[kind] = {"zip": zipname, "sha256": hashlib.sha256(open(zp, "rb").read()).hexdigest(),
                      "help_first_line": first}
json.dump(versions, open("/content/su2_versions.json", "w"), indent=1)
print(json.dumps(versions, indent=1))

SU2_SERIAL = "/content/su2/bin/SU2_CFD"
N_RANKS = max(1, os.cpu_count() // 2)
SU2_CMD = f"mpirun -n {N_RANKS} --allow-run-as-root /content/su2mpi/bin/SU2_CFD"
mpi_ok = N_RANKS > 1
if mpi_ok:
    try:
        t = subprocess.run(["mpirun", "-n", "2", "--allow-run-as-root", "/content/su2mpi/bin/SU2_CFD", "--help"],
                           capture_output=True, text=True, timeout=5, check=False)
        mpi_ok = t.returncode == 0
    except subprocess.TimeoutExpired:
        mpi_ok = False
if not mpi_ok:
    print("MPI not used (one rank or the 5 s mpirun test failed): falling back to the serial binary")
    SU2_CMD = SU2_SERIAL
print("N_RANKS", N_RANKS, "| SU2_CMD", SU2_CMD)

## 3. Google Drive queue folder
Mounts Drive and checks that `su2queue/jobs` and `su2queue/runs` exist (Claude uploads job files to
`jobs/`; the worker writes `runs/`). Creates `su2queue/worker/` for the worker heartbeat.
Look at: no assertion error. If one fires, the Drive account is not the one that owns `su2queue`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
QUEUE = "/content/drive/MyDrive/su2queue"
for d in ("jobs", "runs"):
    assert os.path.isdir(f"{QUEUE}/{d}"), f"{QUEUE}/{d} is missing"
os.makedirs(f"{QUEUE}/worker", exist_ok=True)
print("queue ok:", QUEUE)

## 4. Scaling test (optional)
Skipped unless `SKIP_SCALING = False`. Builds the inlet 8/8 deg medium case with `worker.make_case.make`
on the local disk and runs 200 iterations with 1, 2 and `N_RANKS` ranks through `worker.run_case.run`
(one rank = the serial binary). 200 iterations include start-up, so seconds per iteration are biased high,
but the comparison between rank counts is like for like. Writes `su2queue/runs/scaling_<nproc>.json`.
Run it **before** starting the worker (they would compete for the cores). Look at: the s/it column and
the speed-up against one rank; pick `N_RANKS` for the worker from it.

In [ ]:
SKIP_SCALING = True   # False: run the scaling test (about 3 runs of 200 iterations)
if SKIP_SCALING:
    print("scaling test skipped (SKIP_SCALING = True)")
else:
    import time
    from worker.make_case import make
    from worker.run_case import run
    params = {"family": "inlet2r", "theta1_deg": 8.0, "theta2_deg": 8.0, "solver": "NAVIER_STOKES",
              "mesh_level": "medium", "max_iter": 200, "tag": "scaling"}
    base = "/content/work/scaling"
    shutil.rmtree(base, ignore_errors=True)
    make(params, base)
    rank_list = sorted({1, 2, N_RANKS}) if mpi_ok else [1]
    rows = []
    for n in rank_list:
        wk = f"{base}_n{n}"
        shutil.rmtree(wk, ignore_errors=True)
        shutil.copytree(base, wk)
        cmd = SU2_SERIAL if n == 1 else f"mpirun -n {n} --allow-run-as-root /content/su2mpi/bin/SU2_CFD"
        r = run(wk, cmd, timeout_s=2400)
        its = max(1, r["run"]["iterations"])
        rows.append({"ranks": n, "iterations": r["run"]["iterations"], "exit_code": r["run"]["exit_code"],
                     "wall_time_s": r["run"]["wall_time_s"], "s_per_iter": r["run"]["wall_time_s"] / its})
    for row in rows:
        print(f"ranks={row['ranks']:2d} it={row['iterations']:4d} exit={row['exit_code']} "
              f"wall={row['wall_time_s']:7.1f} s  {row['s_per_iter']:.4f} s/it  "
              f"speed-up vs 1 rank {rows[0]['s_per_iter'] / row['s_per_iter']:.2f}")
    out = {"params": params, "commit": HEAD, "nproc": os.cpu_count(), "rows": rows}
    json.dump(out, open(f"{QUEUE}/runs/scaling_{os.cpu_count()}.json", "w"), indent=1)

## 5. What will run
Lists `jobs/*.json` and the status flags in `runs/<id>/status/`. A job is pending until it has
`2-done` or `2-failed`. A job at `1-running` with an old heartbeat is a crash victim and is run again
by the worker. Look at: the PENDING lines are what the worker will do, oldest first.

In [ ]:
import glob
jobs = sorted(glob.glob(f"{QUEUE}/jobs/*.json"))
pending = 0
for path in jobs:
    cid = os.path.basename(path)[:-len(".json")]
    st = f"{QUEUE}/runs/{cid}/status"
    flags = sorted(f for f in os.listdir(st) if not f.startswith(".")) if os.path.isdir(st) else []
    done = "2-done" in flags or "2-failed" in flags
    pending += not done
    print(f"{'finished' if done else 'PENDING ':8s} {cid:28s} flags: {', '.join(flags) or '-'}")
print(f"{len(jobs)} job file(s), {pending} pending")

## 6. Worker (this cell blocks, which is intended)
Runs `python -m worker.run_worker` in the foreground: it scans `jobs/` every 20 s, runs SU2 on each
pending job (timeout 2400 s) and writes `runs/<id>/` and `runs/<id>.result.json`. While it serves,
the cell shows a spinner and prints one line per event. **Leave this browser tab open.** Colab
disconnects idle or closed sessions; a Colab Pro runtime lasts longer.

- **Stop:** Runtime -> Interrupt execution (Ctrl-C). The worker stops cleanly and flags nothing.
- **Resume:** run this cell again. A job left at `1-running` is re-run once its heartbeat is older than
  10 minutes; finished jobs are never touched. After a runtime reset, run sections 0 to 3 first.
- One worker per queue folder: do not run this in two notebooks at once.

In [ ]:
%cd /content/ai-engineering
!python -m worker.run_worker --queue {QUEUE} --su2-cmd "{SU2_CMD}" --poll 20 --timeout 2400 --worker-id colab-$(hostname)

## What to report back
Paste into the Claude Code session:
1. The lines the worker printed in section 6 (job picked up, finished or failed, and the reason).
2. The list of finished results: run `!ls -l {QUEUE}/runs/*.result.json` in a new cell (it also shows
   `error.json` files with `!ls {QUEUE}/runs/*/error.json`), and say which `<case_id>.result.json` files
   you want downloaded. Claude downloads them from Drive and ingests them with the MCP tools.
3. If you ran the scaling test: the printed table (or `runs/scaling_<nproc>.json`).
4. Anything unusual: MPI falling back to serial, a `mesh_mismatch` (commit differs between the machine
   that staged the job and this notebook), a Colab disconnect.